In [8]:
from ingest_hw import github_data_reader,build_index
from rag_helper_hw import RAGBase
from openai import OpenAI

In [9]:
open_ai_client = OpenAI()

In [10]:
files = github_data_reader()

In [11]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [12]:
print(f"Q1 - How many lesson pages are in the dataset?\n\nANSWER: {len(files)}")

Q1 - How many lesson pages are in the dataset?

ANSWER: 72


In [13]:
index_op = build_index(documents)

In [14]:
index_results = index_op.search(
    "How does the agentic loop keep calling the model until it stops?",
    num_results=5
)
print(f"Q2 - What's the filename of the first result?\n\nANSWER: {index_results[0]['filename']}")

Q2 - What's the filename of the first result?

ANSWER: 01-agentic-rag/lessons/14-agentic-loop.md


In [15]:
assistant = RAGBase(index = index_op,
                    llm_client = open_ai_client)

In [16]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer.output_text)

The loop keeps calling the model by using a `while True` loop and a flag like `has_function_calls`.

Process:
1. Call the model with the current `messages`.
2. If the model returns a `function_call`, run the tool and append the tool output to `messages`.
3. If there were any function calls, repeat the loop.
4. If the response has no function calls, `break` and stop.

So the agent stops when a turn returns a final answer with no more tool calls.


In [17]:
print(f' Q3 - Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?\n\nANWER: {answer.usage.input_tokens}')

 Q3 - Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

ANWER: 7135


In [18]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
print(f"How many chunks do you get?\n\nANSWER: {len(chunks)}")

How many chunks do you get?

ANSWER: 295


In [20]:
#test if input tokens got reduced by using chunks instead of full documents
index_op_chunks = build_index(chunks)
assistant = RAGBase(index = index_op_chunks,
                    llm_client = open_ai_client)
answer_post_chunks = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(f"input tokens after using chunks: {answer_post_chunks.usage.input_tokens}")

input tokens after using chunks: 2318


In [22]:
print(f"Q5 - Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?\n\nANSWER: {answer.usage.input_tokens-answer_post_chunks.usage.input_tokens} fewer input tokens i.e 3x fewer")

Q5 - Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?

ANSWER: 4817 fewer input tokens i.e 3x fewer


In [23]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [24]:
def search(query, num_results=5):
    """
    Search the FAQ database for entries matching the given query.
    """
    return index_op.search(
            query,
            num_results=num_results
        )

In [25]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [26]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'},
    'num_results': {'type': 'string', 'description': 'num_results parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [31]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

In [32]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [33]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


In [49]:

func_call_count = 0
for obj in result.all_messages:
    if  obj.__class__.__name__ == "ResponseFunctionToolCall":
        func_call_count += 1
print(f"Q6 - How many times did the agent call the search tool?\n\nANSWER: {func_call_count} times")

Q6 - How many times did the agent call the search tool?

ANSWER: 7 times
